# FireWatch — quick eval (GPU)
Evaluates the indoor-retrained model on the merged test set + indoor-only set.
**Runtime → Change runtime type → GPU**, then **Runtime → Run all**.

In [ ]:
!nvidia-smi -L
import os
if not os.path.isdir('fire-detection'):
    !git clone --depth 1 -b master https://github.com/01End/fire-detection.git
%cd fire-detection
!pip install -q -U keras-hub opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

ZIP  = '/content/drive/MyDrive/D-Fire.zip'   # the MERGED dataset (has test/ and indoor_test/)
DEST = '/content/data'
if not (os.path.isdir(DEST) and os.listdir(DEST)):
    !mkdir -p {DEST} && unzip -q -o "{ZIP}" -d {DEST}

root = None
for dp, dns, fns in os.walk(DEST):
    if os.path.isdir(os.path.join(dp, 'test', 'images')):
        root = dp; break
assert root, f'no test/images found under {DEST}: {os.listdir(DEST)}'
DATA  = root
MODEL = '/content/drive/MyDrive/fire_retinanet_indoor.weights.h5'  # your retrained model
IMG   = 384   # trained at 384, exposure none
ind = f'{DATA}/indoor_test/images'
print('DATA =', DATA)
print('test:', len(os.listdir(f'{DATA}/test/images')),
      '| indoor_test:', len(os.listdir(ind)) if os.path.isdir(ind) else 'MISSING')

## New model — overall + indoor-only

In [ ]:
!python -m training.tf_eval --model "{MODEL}" --data "{DATA}/test" \
    --image-size {IMG} --score-thr 0.5 --exposure none --limit 500
print('\n===== INDOOR-ONLY (new model) =====')
!python -m training.tf_eval --model "{MODEL}" --data "{DATA}/indoor_test" \
    --image-size {IMG} --score-thr 0.5 --exposure none --limit 750

## Old model on indoor (baseline) — shows the improvement

In [ ]:
OLD = '/content/drive/MyDrive/fire_retinanet_full.weights.h5'
!python -m training.tf_eval --model "{OLD}" --data "{DATA}/indoor_test" \
    --image-size 512 --score-thr 0.5 --exposure none --limit 750

## Threshold sweep — pick the operating point
Same 750/500 images per run (fair comparison). Look for the F1 peak.

In [ ]:
for s in [0.4, 0.5, 0.6, 0.7]:
    print(f'\n########## indoor_test  score-thr={s} ##########')
    !python -m training.tf_eval --model "{MODEL}" --data "{DATA}/indoor_test" \
        --image-size {IMG} --score-thr {s} --exposure none --limit 750

for s in [0.5, 0.6]:
    print(f'\n########## overall test  score-thr={s} ##########')
    !python -m training.tf_eval --model "{MODEL}" --data "{DATA}/test" \
        --image-size {IMG} --score-thr {s} --exposure none --limit 500